In [0]:
from pyspark.sql.functions import date_format, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType

schema = StructType([
    StructField("cnpj_basico", StringType()),
    StructField("cnpj_ordem", StringType()),
    StructField("cnpj_dv", StringType()),
    StructField("identificador_matriz_filial", StringType()),
    StructField("nome_fantasia", StringType()),
    StructField("situacao_cadastral", StringType()),
    StructField("data_situacao_cadastral", StringType()),
    StructField("motivo_situacao_cadastral", StringType()),
    StructField("nome_cidade_exterior", StringType()),
    StructField("pais", StringType()),
    StructField("data_inicio_atividade", StringType()),
    StructField("cnae_principal", StringType()),
    StructField("cnae_secundario", StringType()),
    StructField("tipo_logradouro", StringType()),
    StructField("logradouro", StringType()),
    StructField("numero", StringType()),
    StructField("complemento", StringType()),
    StructField("bairro", StringType()),
    StructField("cep", StringType()),
    StructField("uf", StringType()),
    StructField("municipio", StringType()),
    StructField("ddd1", StringType()),
    StructField("telefone1", StringType()),
    StructField("ddd2", StringType()),
    StructField("telefone2", StringType()),
    StructField("ddd_fax", StringType()),
    StructField("fax", StringType()),
    StructField("email", StringType()),
    StructField("situacao_especial", StringType()),
    StructField("data_situacao_especial", StringType()),
])

(
    spark.read
        .option("sep", ";")
        .schema(schema)
        .csv("/Volumes/rfb/transient/transient/estabelecimento")
    .withColumn("DATA_REF", date_format(current_timestamp(), "yyyy-MM"))
    .withColumn("ingested_at", current_timestamp())
    .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("DATA_REF")
        .saveAsTable("rfb.raw.raw_estabelecimento")
)


count = spark.table('rfb.raw.raw_estabelecimento').count()

if count > 0:
    print("Deleting transient folder.")
    dbutils.fs.rm("/Volumes/rfb/transient/transient/estabelecimento", recurse=True)
else:
    raise Exception("Raw table is empty, please check your source.")